In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
import copy

In [2]:
# 1. Load and Preprocess Data
df = pd.read_csv('diabetes_prediction_dataset.csv')

In [3]:
# Encode categorical features
le = LabelEncoder()
df['gender'] = le.fit_transform(df['gender'])
df['smoking_history'] = le.fit_transform(df['smoking_history'])

# Separate features and target
X = df.drop('diabetes', axis=1)
y = df['diabetes']

In [4]:
# Standardize numerical features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split into Training and Testing sets
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# 2. Simulate Multiple Healthcare Institutions (Clients)
num_clients = 5
client_data_split = np.array_split(np.arange(len(X_train_full)), num_clients)

clients_X = [X_train_full[idx] for idx in client_data_split]
clients_y = [y_train_full.iloc[idx].values for idx in client_data_split]

In [6]:
# 3. Federated Learning Process (FedAvg)
# Initial Global Model (Logistic Regression via SGD)
global_model = SGDClassifier(loss='log_loss', max_iter=1, warm_start=True, random_state=42)

# Initialize global weights by training on a small sample
global_model.fit(X_train_full[:10], y_train_full[:10])

c:\Users\SRINIDHI\anaconda3\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


SGDClassifier(loss='log_loss', max_iter=1, random_state=42, warm_start=True)

In [7]:
num_rounds = 10
print(f"Starting Federated Learning with {num_clients} clients over {num_rounds} rounds...\n")

for round_idx in range(num_rounds):
    local_weights = []
    local_intercepts = []
    
    # Each client trains locally based on the global model
    for client_idx in range(num_clients):
        # Instantiate a local copy of the global model
        local_model = copy.deepcopy(global_model)
        
        # Local training step (one iteration)
        local_model.partial_fit(clients_X[client_idx], clients_y[client_idx], classes=np.unique(y))
        
        # Collect local updates (weights and bias)
        local_weights.append(local_model.coef_)
        local_intercepts.append(local_model.intercept_)
    
    # Aggregation: Averaging the weights (FedAvg)
    avg_weights = np.mean(local_weights, axis=0)
    avg_intercept = np.mean(local_intercepts, axis=0)
    
    # Update the Global Model with the averaged weights
    global_model.coef_ = avg_weights
    global_model.intercept_ = avg_intercept
    
    # Round Evaluation
    y_pred = global_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Round {round_idx + 1}: Global Model Accuracy = {accuracy:.4f}")

Starting Federated Learning with 5 clients over 10 rounds...

Round 1: Global Model Accuracy = 0.9476
Round 2: Global Model Accuracy = 0.9479
Round 3: Global Model Accuracy = 0.9470
Round 4: Global Model Accuracy = 0.9462
Round 5: Global Model Accuracy = 0.9476
Round 6: Global Model Accuracy = 0.9482
Round 7: Global Model Accuracy = 0.9465
Round 8: Global Model Accuracy = 0.9482
Round 9: Global Model Accuracy = 0.9466
Round 10: Global Model Accuracy = 0.9483


In [8]:
# 4. Final Evaluation
y_pred_final = global_model.predict(X_test)
print("\nFinal Global Model Performance Report:")
print(classification_report(y_test, y_pred_final))


Final Global Model Performance Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97     18292
           1       0.68      0.73      0.71      1708

    accuracy                           0.95     20000
   macro avg       0.83      0.85      0.84     20000
weighted avg       0.95      0.95      0.95     20000

